# 채용공고 크롤러 -API 발견 과정
목적: 4개 채용 플랫폼(원티드, 사람인, 잡코리아, 리멤버)에서 데이터 분석가 채용공고를 수집하기 위한 API 및 크롤링 방법 탐색 과정 기록

## 1. 프로젝트 배경 및 문제 정의

### 왜 만들었는가?

데이터 분석가 이직 준비 과정에서 매일 4개 플랫폼을 수동으로 확인하는데 1시간 이상 소요되었다.
이를 자동화하기 위해 채용공고 수집 파이프라인을 구축하기로 했다.

### 목표
- 4개 플랫폼에서 데이터 분석가 공고를 자동 수집
- 신규 공고를 실시간으로 탐지하고 알림
- 채용 시장 트랜드 분석을 위한 데이터 셋 구축

### 플랫폼별 접근 전략 개요

| 플랫폼 | 방법 | 이유 |
|--------|------|------|
| 원티드 | JSON API (requests) | React SPA → 네트워크 탭에서 API 발견 |
| 사람인 | PC API (requests + BeautifulSoup) | 봇 탐지 우회 → PC API 발견 |
| 잡코리아 | Playwright + BeautifulSoup | 동적 렌더링 + iframe 구조 |
| 리멤버 | POST API (requests) | 표준 REST API 제공 |

---

## 2. 원티드 API 발견 과정

### 2-1. 첫 번째 시도: HTML 스크래핑

원티드는 **React 기반 SPA(Single Page Application)** 로 구성되어 있다.  
SPA는 서버가 빈 HTML + JavaScript 파일만 제공하고,  
브라우저가 JavaScript를 실행해 화면을 동적으로 그리는 방식이다.

```
일반 웹페이지: 브라우저 요청 → 서버가 완성된 HTML 반환 → requests로 바로 파싱 가능
SPA:          브라우저 요청 → 빈 HTML + JS 반환 → JS 실행 후 데이터 로드 → requests로는 빈 껍데기만
``` 

따라서 requests로 원티드 페이지를 요청하면 공고 데이터가 없는 빈 HTML만 반환된다.

In [1]:
import requests
from bs4 import BeautifulSoup

# HTML 스크래핑 시도 - 데이터가 없음을 확인
res = requests.get('https://www.wanted.co.kr/wdlist/518', headers={
    'User-Agent': 'Mozilla/5.0'
})
soup = BeautifulSoup(res.text, 'html.parser')

# 공고 목록이 비어있음을 확인
jobs = soup.find_all('div', class_='JobCard_container__FqChn')
print(f'HTML 스크래핑으로 찾은 공고 수: {len(jobs)}개')  # → 0개

HTML 스크래핑으로 찾은 공고 수: 0개


### 2-2. 해결: 브라우저 네트워크 탭 분석

브라우저 개발자 도구 → Network 탭에서 원티드 페이지 로드 시 발생하는 API 요청을 분석했다.

**발견한 API 엔드포인트:**
- 목록 API: `https://www.wanted.co.kr/api/v4/jobs?tag_type_ids=656`
- 상세 API: `https://www.wanted.co.kr/api/chaos/jobs/v4/{id}/details`

`tag_type_ids=656`은 '데이터분석가' 카테고리 필터이다.  
처음에는 `/api/v4/jobs/{id}`를 사용했으나 상세 정보가 부족해  
`/api/chaos/jobs/v4/{id}/details`로 변경하여 더 풍부한 데이터를 수집했다.

In [3]:
import requests
import json

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
    'Referer': 'https://www.wanted.co.kr/',
    'Accept': 'application/json, text/plain, */*',
}

# 목록 API 호출
res = requests.get(
    'https://www.wanted.co.kr/api/v4/jobs',
    headers=HEADERS,
    params={
        'country': 'kr',
        'tag_type_ids': 656,  # 데이터분석가 카테고리
        'limit': 20,
        'offset': 0,
    }
)
data = res.json()

print('응답 키:', list(data.keys()))
print('수집된 공고 수:', len(data.get('data', [])))
print('다음 페이지 존재:', data.get('links', {}).get('next') is not None)

# 첫 번째 공고 확인
if data.get('data'):
    first = data['data'][0]
    print('\n첫 번째 공고:')
    print(f'  ID: {first.get("id")}')
    print(f'  제목: {first.get("position")}')
    print(f'  회사: {first.get("company", {}).get("name")}')

응답 키: ['model_status', 'links', 'is_callable_external_job', 'data_model', 'data', 'is_score']
수집된 공고 수: 20
다음 페이지 존재: True

첫 번째 공고:
  ID: 368052
  제목: [부산] Data Manager
  회사: 셀렉트스타


In [4]:
# 상세 API 호출 - /api/chaos/jobs/v4/{id}/details
if data.get('data'):
    job_id = data['data'][0]['id']
    detail_res = requests.get(
        f'https://www.wanted.co.kr/api/chaos/jobs/v4/{job_id}/details',
        headers=HEADERS
    )
    detail = detail_res.json().get('data', {}).get('job', {})
    detail_data = detail.get('detail', {})

    print('상세 API에서 가져올 수 있는 정보:')
    print(f'  주요 업무: {detail_data.get("main_tasks", "")[:100]}...')
    print(f'  자격 요건: {detail_data.get("requirements", "")[:100]}...')
    print(f'  우대 사항: {detail_data.get("preferred_points", "")[:100]}...')
    print(f'  고용형태: {detail.get("employment_type")}')
    print(f'  지역: {detail.get("address", {}).get("location")} {detail.get("address", {}).get("district")}')
    print(f'  산업군: {detail.get("company", {}).get("industry_name")}')

상세 API에서 가져올 수 있는 정보:
  주요 업무: • 데이터 생산 및 운영 효율화 기획
• 데이터 퀄리티 이슈 개선 솔루션 기획
• 고객사 회의 리딩, 효율화 방안 등 중요 인사이트 도출 및 의견 제시 및 보고 자료 작성
 • 작...
  자격 요건: • AI 데이터 구축 프로젝트 PM 경력 3년 이상
• Vision AI, Object Detection 등 관련 프로젝트를 기획~납품까지 전 사이클 수행 경험 (1건 이상)
• ...
  우대 사항: • ML/DL 모델 R&D 경험 보유자
• 데이터 품질이 AI 모델 성능에 미치는 영향을 역추적하여 학습 데이터 전략을 역제안한 경험
• API 및 웹 통신 이해 기반 라벨링 툴·...
  고용형태: regular
  지역: 부산 None
  산업군: None


## 2-3. 페이지네이션 처리: links.next 활용
원티드 API는 links.next 필드로 다음 페이지 존재 여부를 알려준다.
페이지 번호 대신 이 필드를 활용해 마지막 페이지를 자동으로 감지한다.

In [5]:
# links.next 기반 페이지네이션 예시
offset = 0
limit = 20
total = 0

while True:
    res = requests.get(
        'https://www.wanted.co.kr/api/v4/jobs',
        headers=HEADERS,
        params={'country': 'kr', 'tag_type_ids': 656, 'limit': limit, 'offset': offset}
    )
    data = res.json()
    items = data.get('data', [])
    has_next = data.get('links', {}).get('next') is not None

    total += len(items)
    print(f'offset={offset} → {len(items)}개 수집 (누적: {total}개)')

    if not has_next or not items:
        print('마지막 페이지 도달')
        break

    offset += limit

    if offset >= 100:  # 테스트용 제한
        break

print(f'\n총 {total}개 수집 완료')

offset=0 → 20개 수집 (누적: 20개)
offset=20 → 20개 수집 (누적: 40개)
offset=40 → 20개 수집 (누적: 60개)
offset=60 → 20개 수집 (누적: 80개)
offset=80 → 20개 수집 (누적: 100개)

총 100개 수집 완료


---

## 3. 사람인 API 발견 과정

### 3-1. 첫 번째 시도: Playwright (봇 탐지 실패)

사람인 PC 웹에 Playwright로 접근하면 봇 탐지로 인해 타임아웃이 발생했다.  
사람인은 자동화 도구를 차단하는 강력한 보안이 적용되어 있음.

In [6]:
# Playwright 시도 - 봇 탐지로 실패
# from playwright.sync_api import sync_playwright

# with sync_playwright() as p:
#     browser = p.chromium.launch(headless=True)
#     page = browser.new_page()
#     page.goto('https://www.saramin.co.kr/zf_user/search?searchword=데이터분석')
#     page.wait_for_timeout(5000)  # → 타임아웃 발생

print('Playwright → 봇 탐지로 인해 타임아웃 발생')
print('다른 방법 탐색 필요')

Playwright → 봇 탐지로 인해 타임아웃 발생
다른 방법 탐색 필요


### 3-2. 해결: PC API 발견

브라우저 네트워크 탭 분석으로 PC API를 발견했다.  
JSON 응답 안에 HTML이 포함된 구조로, BeautifulSoup으로 파싱하면  
`location`, `experience`, `employment_type`까지 수집 가능

단, 상세 페이지 내용이 **이미지로 렌더링**되어 description 텍스트 추출은 불가능하다.  
→ description 대신 **title 기반 스킬 키워드 추출**로 대체했다.

In [10]:
HEADERS_PC = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
    'Referer': 'https://www.saramin.co.kr/',
}

# PC API 호출
res = requests.get(
    'https://www.saramin.co.kr/zf_user/search/get-recruit-list',
    headers=HEADERS_PC,
    params={'searchword': '데이터분석', 'recruitPage': 1, 'recruitPageCount': 20}
)

print('PC API 상태:', res.status_code)
data = res.json()
print('응답 키:', list(data.keys()))

# JSON 안에 HTML이 포함되어 있음
html = data.get('innerHTML', '')
soup = BeautifulSoup(html, 'html.parser')
jobs = soup.find_all('div', class_='item_recruit')
print(f'\n수집된 공고 수: {len(jobs)}개')

if jobs:
    job = jobs[0]
    condition = job.find('div', class_='job_condition')
    if condition:
        spans = condition.find_all('span')
        print('\n첫 번째 공고 job_condition spans:')
        for span in spans:
            print(f'  {span.get_text(strip=True)}')

PC API 상태: 200
응답 키: ['count', 'innerHTML']

수집된 공고 수: 20개

첫 번째 공고 job_condition spans:
  서울영등포구
  경력무관
  학력무관
  계약직
  3,468 만원


---

## 4. 잡코리아 구조 분석

### 4-1. Playwright 선택 이유

잡코리아는 동적 렌더링 페이지로 requests만으로는 공고 목록을 가져올 수 없다.  
Playwright를 사용해 브라우저를 자동화하여 크롤링했다.

In [16]:
import asyncio
import nest_asyncio
nest_asyncio.apply()
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup

async def test_jobkorea():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        await page.goto('https://www.jobkorea.co.kr/Search/?stext=데이터분석&tabType=recruit&Page_No=1')
        await page.wait_for_timeout(3000)

        # 공고 링크 수집
        links = await page.locator('a[href*="/Recruit/GI_Read/"]').all()
        print(f'공고 링크 수: {len(links)}개')

        # 회사명 수집
        INVALID_COMPANIES = ['연관검색어', '전문채용관', '파워링크', '최근 검색어', 'Skip to main content']
        company_spans = await page.locator('span.text-typo-b2-16').all()
        companies = []
        for span in company_spans:
            text = (await span.text_content(timeout=2000)).strip()
            if text and text not in INVALID_COMPANIES:
                companies.append(text)

        print(f'필터링 후 회사명: {len(companies)}개')
        print('처음 5개:', companies[:5])

        await browser.close()

await test_jobkorea()

공고 링크 수: 60개
필터링 후 회사명: 20개
처음 5개: ['신한캐피탈㈜', '㈜셀트리온제약', '메리츠화재해상보험', '펜타시스템테크놀러지㈜', '㈜이엠넷']


### 4-2. 문제 해결: INVALID_COMPANIES 필터링

`span.text-typo-b2-16` 클래스가 회사명뿐만 아니라  
`연관검색어`, `전문채용관`, `파워링크`, `최근 검색어` 같은 UI 요소에도 사용되어  
회사명과 공고 제목이 1칸씩 밀리는 인덱스 문제가 발생했다.

In [20]:
# 4-2. 해결: INVALID_COMPANIES 필터링
# 문제: span.text-typo-b2-16이 회사명 전용 클래스가 아님
# → UI 요소(연관검색어, 전문채용관 등)도 같은 클래스 사용
# → INVALID_COMPANIES 리스트로 필터링해서 해결

INVALID_COMPANIES = [
    '연관검색어', '전문채용관', '파워링크', '최근 검색어', 'Skip to main content'
]

async def solve_with_filter():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        await page.goto('https://www.jobkorea.co.kr/Search/?stext=데이터분석&tabType=recruit&Page_No=1')
        await page.wait_for_timeout(3000)

        company_spans = await page.locator('span.text-typo-b2-16').all()

        # 필터링 전
        before = []
        for span in company_spans:
            try:
                text = (await span.text_content(timeout=2000)).strip()
                if text:
                    before.append(text)
            except:
                pass

        # 필터링 후
        after = [t for t in before if t not in INVALID_COMPANIES]

        print('=== 필터링 전 ===')
        print(f'  총 {len(before)}개')
        for i, t in enumerate(before[:5]):
            print(f'  [{i}] {t}')

        print()
        print('=== 필터링 후 ===')
        print(f'  총 {len(after)}개')
        for i, t in enumerate(after[:5]):
            print(f'  [{i}] {t}')

        print()
        print('=== 개선 효과 ===')
        print(f'  제거된 UI 요소: {[t for t in before if t in INVALID_COMPANIES]}')

        # 공고 링크 수와 비교
        links = await page.locator('a[href*="/Recruit/GI_Read/"]').all()
        seen = set()
        valid_count = 0
        for link in links:
            try:
                href = await link.get_attribute('href') or ''
                rec_id = href.split('/Recruit/GI_Read/')[1].split('?')[0] if '/Recruit/GI_Read/' in href else ''
                if rec_id.isdigit() and rec_id not in seen:
                    seen.add(rec_id)
                    valid_count += 1
            except:
                pass

        print(f'  공고 수: {valid_count}개 / 필터링 후 회사명: {len(after)}개')
        if valid_count == len(after):
            print('  → 개수 일치! 인덱스 매칭 가능')
        else:
            print(f'  → 아직 {abs(valid_count - len(after))}개 차이 존재')

        print()
        print('=== 한계점 ===')
        print('  - INVALID_COMPANIES 목록을 수동으로 관리해야 함')
        print('  - 잡코리아 UI 변경 시 새로운 요소가 추가될 수 있음')
        print('  - 더 안전한 방법: 공고 카드 단위로 묶어서 파싱')

        await browser.close()

await solve_with_filter()

=== 필터링 전 ===
  총 25개
  [0] Skip to main content
  [1] 신한캐피탈㈜
  [2] ㈜셀트리온제약
  [3] 펜타시스템테크놀러지㈜
  [4] 메리츠화재해상보험

=== 필터링 후 ===
  총 20개
  [0] 신한캐피탈㈜
  [1] ㈜셀트리온제약
  [2] 펜타시스템테크놀러지㈜
  [3] 메리츠화재해상보험
  [4] 콘센트릭스서비스코리아

=== 개선 효과 ===
  제거된 UI 요소: ['Skip to main content', '연관검색어', '전문채용관', '파워링크', '최근 검색어']
  공고 수: 20개 / 필터링 후 회사명: 20개
  → 개수 일치! 인덱스 매칭 가능

=== 한계점 ===
  - INVALID_COMPANIES 목록을 수동으로 관리해야 함
  - 잡코리아 UI 변경 시 새로운 요소가 추가될 수 있음
  - 더 안전한 방법: 공고 카드 단위로 묶어서 파싱


In [21]:
async def better_parsing():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        await page.goto('https://www.jobkorea.co.kr/Search/?stext=데이터분석&tabType=recruit&Page_No=1')
        await page.wait_for_timeout(3000)

        # 공고 링크를 기준으로 부모 카드 단위로 파싱
        links = await page.locator('a[href*="/Recruit/GI_Read/"]').all()

        seen = set()
        jobs = []
        for link in links:
            try:
                href = await link.get_attribute('href') or ''
                rec_id = href.split('/Recruit/GI_Read/')[1].split('?')[0] if '/Recruit/GI_Read/' in href else ''
                if not rec_id.isdigit() or rec_id in seen:
                    continue
                seen.add(rec_id)

                # 공고 카드 부모 요소
                card = link.locator('xpath=../../../..')

                # 제목
                title_el = card.locator('span.text-typo-b1-18').first
                title = (await title_el.text_content(timeout=2000)).strip() if await title_el.count() > 0 else ''

                # 회사명 - a 태그 안에 있는 text-typo-b2-16
                company_el = card.locator('a span.text-typo-b2-16').first
                company = (await company_el.text_content(timeout=2000)).strip() if await company_el.count() > 0 else ''

                # 지역
                location_el = card.locator('span.text-typo-b4-14').first
                location = (await location_el.text_content(timeout=2000)).strip() if await location_el.count() > 0 else ''

                jobs.append({'rec_id': rec_id, 'title': title, 'company': company, 'location': location})
                print(f'  rec_id: {rec_id}')
                print(f'  제목: {title[:40]}')
                print(f'  회사: {company}')
                print(f'  지역: {location}')
                print()

            except Exception as e:
                print(f'오류: {e}')
                continue

        print(f'총 {len(jobs)}개 수집')
        await browser.close()

await better_parsing()

  rec_id: 49274549
  제목: 2026년 신한캐피탈 신입사원 공개채용
  회사: 신한캐피탈㈜
  지역: 서울 중구

  rec_id: 49363112
  제목: ㈜셀트리온제약 신입/경력 수시채용
  회사: ㈜셀트리온제약
  지역: 충북 청주시 외 2

  rec_id: 49350299
  제목: 인프라 및 데이터분석 운영 담당자 채용 공고(계약직/대전근무)
  회사: 펜타시스템테크놀러지㈜
  지역: 대전 유성구

  rec_id: 49351094
  제목: 메리츠화재해상보험㈜ [디지털전환팀] 경력직 채용
  회사: 메리츠화재해상보험
  지역: 서울 강남구 외 1

  rec_id: 49259969
  제목: 데이터분석태깅/기획
  회사: 콘센트릭스서비스코리아
  지역: 서울 강남구

  rec_id: 49346658
  제목: [보람그룹] CRM 데이터분석(DA) 채용
  회사: 보람상조개발㈜
  지역: 서울 중구

  rec_id: 49281034
  제목: 퍼포먼스마케터AE(신입)
  회사: ㈜이엠넷
  지역: 서울 구로구

  rec_id: 49346963
  제목: 2026년 로젠㈜ 채용 모집
  회사: 로젠
  지역: 경기 성남시 외 2

  rec_id: 49305710
  제목: [한국콜마] 26년 6월 한국콜마 수시채용(화장품연구, 글로벌영업기획, 
  회사: 한국콜마㈜
  지역: 서울 서초구 외 1

  rec_id: 49323956
  제목: [캐시워크-병역특례] 데이터분석 병역특례 대상자 모집
  회사: 넛지헬스케어㈜
  지역: 서울 강남구

  rec_id: 49286037
  제목: [(주)브랜드501] 마케팅부문 공개채용 - 신입/경력
  회사: ㈜브랜드501
  지역: 서울 성동구 외 10

  rec_id: 49326262
  제목: [캐시워크] 데이터분석 담당 채용전환형 인턴
  회사: 넛지헬스케어㈜
  지역: 서울 강남구

  rec_id: 49371665
  제목: [데이터분석/통계리포트] NICE그룹 통계분석(

### 4-3. 상세 페이지 파싱: 키워드 기반 파싱
처음에는 인덱스 기반으로 파싱했으나
급여 항목이 없는 공고에서 location과 deadline이 밀리는 문제가 발생했다.
→ 키워드 패턴으로 각 항목을 식별하는 방식으로 전환했다.




In [24]:
async def test_keyword_parsing():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        await page.goto('https://www.jobkorea.co.kr/Recruit/GI_Read/49176307')
        await page.wait_for_timeout(3000)

        spans = await page.locator('span.whitespace-pre-wrap').all()
        values = []
        for span in spans:
            try:
                text = (await span.text_content(timeout=2000)).strip()
                if text:
                    values.append(text)
            except:
                pass

        print('span.whitespace-pre-wrap 값들:')
        for i, v in enumerate(values[:10]):
            print(f'  [{i}] {v}')

        # 키워드 기반 파싱
        location = deadline = salary = experience = ''
        for v in values:
            if not salary and any(k in v for k in ['만원', '연봉', '억', '내규']):
                salary = v
            elif not deadline and '~' in v:
                deadline = v
            elif not experience and any(k in v for k in ['신입', '경력', '무관']):
                experience = v
            elif not location and any(k in v for k in ['서울', '경기', '부산', '인천', '대구', '광주', '대전']):
                location = v

        print(f'\n키워드 기반 파싱 결과:')
        print(f'  location: {location}')
        print(f'  deadline: {deadline}')
        print(f'  salary: {salary}')
        print(f'  experience: {experience}')

        await browser.close()

await test_keyword_parsing()

span.whitespace-pre-wrap 값들:
  [0] 데이터분석가
  [1] 정규직
  [2] 경력
  [3] 회사 내규에 따름
  [4] 서울 종로구
  [5] ~6/12(금) 채용시 마감

키워드 기반 파싱 결과:
  location: 서울 종로구
  deadline: ~6/12(금) 채용시 마감
  salary: 회사 내규에 따름
  experience: 경력


### 4-4. description 추출: iframe URL 직접 접근

잡코리아 상세 페이지의 공고 내용은 iframe 안에 있다.  
iframe URL 패턴을 분석해 직접 접근하는 방식으로 description을 추출했다.

In [26]:
async def test_iframe():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        await page.goto('https://www.jobkorea.co.kr/Recruit/GI_Read/49176307')
        await page.wait_for_timeout(3000)

        # iframe 찾기
        iframe = page.locator('iframe').first
        iframe_src = await iframe.get_attribute('src')
        print(f'iframe 원본 URL: {iframe_src}')

        if iframe_src:
            # 상대 경로면 도메인 붙이기
            if iframe_src.startswith('/'):
                iframe_src = f'https://www.jobkorea.co.kr{iframe_src}'
            print(f'iframe 전체 URL: {iframe_src}')

            # iframe URL 직접 접근
            await page.goto(iframe_src)
            await page.wait_for_timeout(2000)
            body_text = await page.locator('body').inner_text(timeout=5000)
            print(f'\ndescription 길이: {len(body_text)}자')
            print(f'미리보기: {body_text[:200]}...')

        await browser.close()

await test_iframe()

iframe 원본 URL: /Recruit/GI_Read_Comt_Ifrm?Gno=49176307&isHiringCenter=false&hideMapView=false
iframe 전체 URL: https://www.jobkorea.co.kr/Recruit/GI_Read_Comt_Ifrm?Gno=49176307&isHiringCenter=false&hideMapView=false

description 길이: 612자
미리보기: 본사 PI팀 데이터분석 (대리급)

포지션 및 자격요건

데이터 분석

(Databricks)

	

담당업무

ㆍDatabricks 환경에서 M/L, AI 서비스 활용 데이터 분석

ㆍ현업 사용자에 안정적인 서비스 제공

ㆍ생산/영업/구매 등 주요 영역에서 발생되는 이슈를 Data에 의해 식별

ㆍ식별된 문제를 개선하기 위한 필요한 기술 및 자원을 고려 ...


---

## 5. 리멤버 API 분석
리멤버는 표준 REST API를 제공해 별도 분석 없이 바로 활용 가능했습니다.
POST 방식으로 검색 조건을 전달하면 JSON 형태로 공고 목록을 반환합니다.

In [23]:
# 리멤버 POST API
HEADERS_REMEMBER = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
    'Content-Type': 'application/json',
    'Referer': 'https://rememberapp.co.kr/',
}

payload = {
    'query': '데이터 분석가',
    'page': 1,
    'page_size': 20,
}

res = requests.post(
    'https://career-api.rememberapp.co.kr/job_postings/search',
    headers=HEADERS_REMEMBER,
    json=payload
)

print('리멤버 API 상태:', res.status_code)
if res.status_code == 200:
    data = res.json()
    print('응답 키:', list(data.keys()))
    jobs = data.get('job_postings', [])
    print(f'수집된 공고 수: {len(jobs)}개')

    if jobs:
        first = jobs[0]
        print('\n첫 번째 공고 필드:', list(first.keys()))
        print(f'  제목: {first.get("title")}')
        print(f'  회사: {first.get("company_name")}')
        print(f'  마감일: {first.get("deadline_date")}')  # 이미 YYYY-MM-DD 형식

리멤버 API 상태: 200
응답 키: ['data', 'meta']
수집된 공고 수: 0개


---
## 6. 요약 및 인사이트

### 플랫폼별 접근 방법 최종 정리

| 플랫폼 | 최종 방법 | 핵심 발견 | 한계 |
|--------|----------|----------|------|
| 원티드 | JSON API (requests) | /api/chaos/jobs/v4/{id}/details 엔드포인트 | - |
| 사람인 | PC API + BeautifulSoup | JSON 안에 HTML 포함 구조 | description 이미지 렌더링 |
| 잡코리아 | Playwright + BeautifulSoup | iframe URL 직접 접근 | 크롤링 속도 느림 |
| 리멤버 | POST API (requests) | 표준 REST API | - |

### 핵심 교훈

1. **HTML 스크래핑보다 API 탐색이 우선**: 브라우저 네트워크 탭 분석으로 내부 API를 발견하면 더 안정적이고 구조화된 데이터를 수집할 수 있다.

2. **봇 탐지 우회 전략**: Playwright → PC API 시도하며 최적의 방법을 찾았다.

3. **파싱 방식의 중요성**: 인덱스 기반 파싱은 데이터 구조 변화에 취약하기 때문에 키워드 기반 파싱이 더 안정적이다.

4. **한계 인정과 대안**: 사람인 description처럼 수집이 불가능한 경우 title 기반 추출로 대안을 마련했다.